# 5.1 Production RAG with Evaluation

**Measure and improve your RAG pipeline with real metrics.**

In this notebook you will:
- Build a complete RAG pipeline (reusing patterns from previous notebooks)
- Create a **golden test set** of question-answer pairs
- Evaluate the pipeline using RAGAS-style metrics
- Interpret scores and identify what to fix
- Improve the pipeline and re-evaluate

> **Requirements:** Groq API key from [console.groq.com](https://console.groq.com/keys)

## 1. Setup

In [ ]:
# Install all required packages:
# - ragas: the RAG evaluation framework
# - datasets: required by ragas for data handling
# - matplotlib: for visualizing evaluation results
!pip install langchain langchain-groq langchain-community chromadb sentence-transformers ragas datasets matplotlib -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. Build the RAG Pipeline

We'll build a complete pipeline using the patterns from notebooks 3.1 and 4.1. This is the pipeline we'll evaluate.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- Knowledge Base ---
documents_text = [
    {
        "text": """CANCELLATION AND REFUND POLICY
All tickets can be cancelled within 24 hours of booking for a full refund, regardless of fare class.
After 24 hours, cancellation fees apply:
- Economy: $75 cancellation fee, refund within 7-10 business days
- Premium Economy: $50 cancellation fee, refund within 5-7 business days
- Business Class: Free cancellation up to 48 hours before departure
- First Class: Free cancellation up to 24 hours before departure
Non-refundable tickets cannot be cancelled but can be changed for a $150 change fee plus fare difference.
Travel insurance provides full refund coverage for medical and family emergencies.""",
        "source": "cancellation_policy"
    },
    {
        "text": """BAGGAGE POLICY
Carry-on: One personal item and one carry-on bag (max 55x40x20cm, 7kg).
Checked baggage — Domestic: Economy 1 bag 23kg, Business 2 bags 32kg each.
Checked baggage — International: Economy 2 bags 23kg each, Premium Economy 2 bags 28kg each,
Business 3 bags 32kg each, First Class 3 bags 32kg each plus 1 garment bag.
Excess baggage: $50 per additional bag, $100 for overweight (23-32kg).
Sports equipment counts as one checked bag if under 23kg.
Fragile items should be carried in cabin — airline not liable for checked fragile items.""",
        "source": "baggage_policy"
    },
    {
        "text": """LOYALTY PROGRAM - SKYREWARDS
Earning: 1 mile/km (economy), 1.5x (premium economy), 2x (business), 3x (first class).
Partner hotels: 500 miles/night. Car rentals: 250 miles/rental. Credit card: 1 mile/$1.
Tiers: Silver (25k miles) — priority check-in, 1 free bag.
Gold (50k miles) — lounge access, priority boarding, 2 free bags, free seat selection.
Platinum (100k miles) — all Gold + free upgrades, annual companion ticket.
Redemption: Domestic from 10k miles, International from 25k miles, Upgrades from 5k miles.
Miles expire after 24 months of account inactivity.""",
        "source": "loyalty_program"
    },
    {
        "text": """BOOKING RULES
Online booking available 330 days before departure.
Tickets must be purchased within 24 hours of reservation or booking is auto-cancelled.
Seat selection: Free for Business/First. Economy $15-45 depending on seat type.
Special meals (vegetarian, halal, kosher, gluten-free): request 48 hours before departure.
Unaccompanied minors (5-14): $100 escort service per segment. Under 5 cannot fly alone.
Wheelchair assistance: free, request 48 hours in advance.
Name changes not allowed. Spelling corrections (up to 3 chars) are free.
Group bookings (10+): 10% discount, flexible payment.""",
        "source": "booking_rules"
    },
    {
        "text": """FLIGHT DELAY AND CANCELLATION COMPENSATION
Delays over 3 hours: meal vouchers ($15 per meal period).
Delays over 6 hours or overnight: hotel accommodation at partner hotels provided.
Cancelled flights: free rebooking on next available flight or full refund.
EU flights over 3500km delayed 4+ hours: EUR 600 compensation per EU regulation.
EU flights 1500-3500km delayed 3+ hours: EUR 400 compensation.
EU flights under 1500km delayed 3+ hours: EUR 250 compensation.
Passengers can claim compensation within 6 months of the affected flight.
Weather-related delays are not eligible for monetary compensation but rebooking is free.""",
        "source": "compensation_policy"
    },
]

# --- Build Pipeline ---
# Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_chunks = []
for raw_doc in documents_text:
    doc = Document(page_content=raw_doc["text"], metadata={"source": raw_doc["source"]})
    chunks = text_splitter.split_documents([doc])
    all_chunks.extend(chunks)

# Create embeddings and vector store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name="production_rag"
)

# Create retriever, LLM, prompt, and chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful airline customer service assistant. Answer the question based ONLY on the context below.
If the context doesn't contain the answer, say "I don't have information about that."
Be specific and cite numbers/policies when available.

Context:
{context}

Question: {question}

Answer:
""")

parser = StrOutputParser()


def format_docs(docs):
    """Join retrieved document texts with double newlines."""
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | parser
)

print(f"Pipeline ready: {len(all_chunks)} chunks indexed from {len(documents_text)} documents")

## 3. Create a Golden Test Set

To evaluate a RAG pipeline, you need a **golden test set** — a set of questions with **known correct answers**.

Each test case has:
- **question**: what the user asks
- **ground_truth**: the correct answer (verified against our source documents)
- **source**: which document contains the answer

Creating good test sets is critical. The quality of your evaluation is only as good as your ground truth.

In [ ]:
# Golden test set: 5 question-answer pairs with verified ground truth answers.
# Each answer was written by reading the source documents carefully.
golden_test_set = [
    {
        "question": "What is the cancellation fee for an economy ticket after 24 hours?",
        "ground_truth": "The cancellation fee for an economy ticket after 24 hours is $75, with a refund processed within 7-10 business days.",
        "source": "cancellation_policy"
    },
    {
        "question": "How many checked bags can I bring on an international business class flight?",
        "ground_truth": "On international business class flights, you can bring 3 checked bags of 32kg each.",
        "source": "baggage_policy"
    },
    {
        "question": "What benefits does Gold loyalty status provide?",
        "ground_truth": "Gold status (50,000 miles/year) provides lounge access, priority boarding, 2 free checked bags, and free seat selection.",
        "source": "loyalty_program"
    },
    {
        "question": "Can a 6-year-old child fly alone?",
        "ground_truth": "Yes, children aged 5-14 can fly unaccompanied using the airline's escort service for $100 per flight segment.",
        "source": "booking_rules"
    },
    {
        "question": "What compensation do I get if my EU flight over 3500km is delayed 5 hours?",
        "ground_truth": "For EU flights over 3500km delayed more than 4 hours, you are entitled to EUR 600 compensation. You also receive meal vouchers ($15 per meal period) for delays over 3 hours.",
        "source": "compensation_policy"
    },
]

print(f"Golden test set: {len(golden_test_set)} question-answer pairs")
for i, test in enumerate(golden_test_set, 1):
    print(f"  {i}. {test['question'][:60]}... (source: {test['source']})")

## 4. Generate Answers and Collect Contexts

Before we can evaluate, we run each question through our pipeline and collect:
- The **generated answer** (what the RAG pipeline outputs)
- The **retrieved contexts** (what documents were used)

In [ ]:
# Run each test question through the pipeline and collect results
results = []

for test_case in golden_test_set:
    question = test_case["question"]

    # Get the answer from our RAG chain
    answer = rag_chain.invoke(question)

    # Get the retrieved contexts (separately, so we can evaluate retrieval)
    retrieved_docs = retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]

    results.append({
        "question": question,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": test_case["ground_truth"]
    })

    print(f"Q: {question[:60]}...")
    print(f"A: {answer[:120]}...")
    print()

## 5. Understanding RAGAS Metrics

**RAGAS** (Retrieval-Augmented Generation Assessment) evaluates RAG pipelines with four key metrics:

| Metric | What it measures | Good score | If low, fix by... |
|--------|-----------------|------------|--------------------|
| **Faithfulness** | Is the answer supported by the context? (No hallucination) | > 0.8 | Stricter prompt, lower temperature |
| **Answer Relevancy** | Does the answer actually address the question? | > 0.8 | Better prompt instructions |
| **Context Precision** | Are the retrieved documents relevant? | > 0.7 | Smaller chunks, add re-ranking |
| **Context Recall** | Do the retrieved docs contain all needed info? | > 0.7 | Increase k, try HyDE |

Each metric tells you about a **different component**: faithfulness/relevancy diagnose the **LLM**, while precision/recall diagnose the **retriever**.

## 6. Evaluate with LLM-as-Judge

We use the LLM itself to judge the quality of each answer. This is the same approach RAGAS uses internally. We implement it manually for reliability with Groq.

In [ ]:
# Create a dedicated evaluator LLM
eval_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


def evaluate_faithfulness(question: str, answer: str, context: str) -> float:
    """Faithfulness: is the answer supported by the context? (0-1)
    High = answer ONLY contains information from the context.
    Low = answer contains hallucinated information not in the context."""
    prompt = ChatPromptTemplate.from_template(
        """Given the context and the answer, rate how faithful the answer is to the context.
A faithful answer ONLY contains information that can be verified from the context.
An unfaithful answer adds information not present in the context (hallucination).

Context:
{context}

Answer:
{answer}

Rate faithfulness from 0.0 to 1.0. Respond with ONLY a number, nothing else."""
    )
    chain = prompt | eval_llm | StrOutputParser()
    result = chain.invoke({"context": context, "answer": answer})
    try:
        return min(1.0, max(0.0, float(result.strip())))
    except ValueError:
        return 0.5  # Default if parsing fails


def evaluate_relevancy(question: str, answer: str) -> float:
    """Answer Relevancy: does the answer address the question? (0-1)
    High = answer directly addresses what was asked.
    Low = answer is off-topic or doesn't answer the question."""
    prompt = ChatPromptTemplate.from_template(
        """Rate how relevant this answer is to the question.
A relevant answer directly addresses what was asked.
An irrelevant answer talks about something else or misses the point.

Question: {question}
Answer: {answer}

Rate relevancy from 0.0 to 1.0. Respond with ONLY a number, nothing else."""
    )
    chain = prompt | eval_llm | StrOutputParser()
    result = chain.invoke({"question": question, "answer": answer})
    try:
        return min(1.0, max(0.0, float(result.strip())))
    except ValueError:
        return 0.5


def evaluate_context_precision(question: str, contexts: list[str]) -> float:
    """Context Precision: are the retrieved documents relevant to the question? (0-1)
    High = retrieved docs contain information needed to answer the question.
    Low = retrieved docs are about unrelated topics."""
    context_text = "\n---\n".join(contexts)
    prompt = ChatPromptTemplate.from_template(
        """Given the question, rate how relevant the retrieved documents are.
Do these documents contain information needed to answer the question?

Question: {question}
Retrieved Documents:
{contexts}

Rate precision from 0.0 to 1.0. Respond with ONLY a number, nothing else."""
    )
    chain = prompt | eval_llm | StrOutputParser()
    result = chain.invoke({"question": question, "contexts": context_text})
    try:
        return min(1.0, max(0.0, float(result.strip())))
    except ValueError:
        return 0.5


def evaluate_context_recall(ground_truth: str, contexts: list[str]) -> float:
    """Context Recall: do retrieved docs contain the information from ground truth? (0-1)
    High = all key facts from the ground truth can be found in the retrieved docs.
    Low = important information from the ground truth is missing."""
    context_text = "\n---\n".join(contexts)
    prompt = ChatPromptTemplate.from_template(
        """Compare the ground truth answer with the retrieved documents.
How much of the ground truth information is present in the retrieved documents?

Ground Truth: {ground_truth}
Retrieved Documents:
{contexts}

Rate recall from 0.0 to 1.0. Respond with ONLY a number, nothing else."""
    )
    chain = prompt | eval_llm | StrOutputParser()
    result = chain.invoke({"ground_truth": ground_truth, "contexts": context_text})
    try:
        return min(1.0, max(0.0, float(result.strip())))
    except ValueError:
        return 0.5


print("Four evaluation functions defined:")
print("  - faithfulness:      is the answer grounded in context?")
print("  - relevancy:         does the answer address the question?")
print("  - context_precision: are retrieved docs relevant?")
print("  - context_recall:    do retrieved docs have all needed info?")

In [ ]:
# Run evaluation on all test questions.
# The LLM judges each metric — this takes a minute.
print("Evaluating the RAG pipeline...\n")

all_scores = []

for i, result in enumerate(results, 1):
    q = result["question"]
    a = result["answer"]
    c = result["contexts"]
    gt = result["ground_truth"]
    context_str = "\n".join(c)

    # Compute all four metrics
    faith = evaluate_faithfulness(q, a, context_str)
    rel = evaluate_relevancy(q, a)
    prec = evaluate_context_precision(q, c)
    rec = evaluate_context_recall(gt, c)

    scores = {
        "question": q[:50] + "...",
        "faithfulness": faith,
        "relevancy": rel,
        "precision": prec,
        "recall": rec,
    }
    all_scores.append(scores)

    print(f"Q{i}: {q[:50]}...")
    print(f"  Faithfulness: {faith:.2f}  Relevancy: {rel:.2f}  "
          f"Precision: {prec:.2f}  Recall: {rec:.2f}")

print("\nEvaluation complete!")

## 7. Interpret Scores and Diagnose Issues

In [ ]:
# Calculate average scores across all test questions
avg_faith = sum(s["faithfulness"] for s in all_scores) / len(all_scores)
avg_rel = sum(s["relevancy"] for s in all_scores) / len(all_scores)
avg_prec = sum(s["precision"] for s in all_scores) / len(all_scores)
avg_rec = sum(s["recall"] for s in all_scores) / len(all_scores)

print("=" * 60)
print("RAG PIPELINE EVALUATION REPORT")
print("=" * 60)
print(f"\n  Faithfulness:      {avg_faith:.2f}  {'PASS' if avg_faith >= 0.8 else 'NEEDS WORK'}")
print(f"  Answer Relevancy:  {avg_rel:.2f}  {'PASS' if avg_rel >= 0.8 else 'NEEDS WORK'}")
print(f"  Context Precision: {avg_prec:.2f}  {'PASS' if avg_prec >= 0.7 else 'NEEDS WORK'}")
print(f"  Context Recall:    {avg_rec:.2f}  {'PASS' if avg_rec >= 0.7 else 'NEEDS WORK'}")

# Diagnose the weakest areas
print("\n--- Diagnosis ---")
if avg_faith < 0.8:
    print("LOW FAITHFULNESS: The LLM is hallucinating.")
    print("  Fix: Strengthen the prompt ('ONLY use the provided context')")
    print("  Fix: Lower temperature to 0")
if avg_rel < 0.8:
    print("LOW RELEVANCY: Answers don't address the questions.")
    print("  Fix: Improve the system prompt with clearer instructions")
if avg_prec < 0.7:
    print("LOW PRECISION: Retriever is fetching irrelevant documents.")
    print("  Fix: Try smaller chunk sizes for more focused retrieval")
    print("  Fix: Add re-ranking (cross-encoder) after retrieval")
if avg_rec < 0.7:
    print("LOW RECALL: Retriever is missing relevant documents.")
    print("  Fix: Increase k (retrieve more documents)")
    print("  Fix: Try HyDE for better query-document matching")
if all(s >= 0.7 for s in [avg_faith, avg_rel, avg_prec, avg_rec]):
    print("All metrics at acceptable levels. Fine-tune for further improvement.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize scores per question to spot problem areas
questions_short = [f"Q{i+1}" for i in range(len(all_scores))]
metrics = ["faithfulness", "relevancy", "precision", "recall"]
colors = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

# Create a grouped bar chart
x = np.arange(len(questions_short))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))

for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = [s[metric] for s in all_scores]
    ax.bar(x + i * width, values, width, label=metric.title(), color=color)

# Threshold line
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Threshold (0.7)")

ax.set_xlabel("Test Question")
ax.set_ylabel("Score")
ax.set_title("RAG Pipeline Evaluation — Scores by Question")
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(questions_short)
ax.set_ylim(0, 1.1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Print the question mapping
print("\nQuestion key:")
for i, s in enumerate(all_scores):
    print(f"  Q{i+1}: {s['question']}")

## 8. Improve the Pipeline

Based on the evaluation, let's apply improvements:
1. **Better chunking** — larger chunks for more context
2. **Better prompt** — stricter instructions against hallucination
3. **More context** — increase k from 4 to 6

Then re-evaluate to measure the improvement.

In [ ]:
# Improvement 1: Larger chunks keep more context together
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Was 300 — larger chunks preserve more context
    chunk_overlap=100     # Was 50 — more overlap prevents cutting sentences
)

improved_chunks = []
for raw_doc in documents_text:
    doc = Document(page_content=raw_doc["text"], metadata={"source": raw_doc["source"]})
    chunks = improved_splitter.split_documents([doc])
    improved_chunks.extend(chunks)

# New vector store
improved_vectorstore = Chroma.from_documents(
    documents=improved_chunks,
    embedding=embedding_model,
    collection_name="production_rag_improved"
)

# Improvement 2: Retrieve more documents (k=6)
improved_retriever = improved_vectorstore.as_retriever(search_kwargs={"k": 6})

# Improvement 3: Stricter prompt
improved_prompt = ChatPromptTemplate.from_template("""
You are an airline customer service assistant. Answer based ONLY on the context.

RULES:
- Include specific numbers, fees, and conditions from the context
- If the context doesn't contain the answer, say "I don't have that information"
- Do NOT add information not explicitly stated in the context
- Be concise but complete

Context:
{context}

Question: {question}

Answer:
""")

# Build improved chain
improved_chain = (
    {"context": improved_retriever | format_docs, "question": RunnablePassthrough()}
    | improved_prompt
    | llm
    | parser
)

print(f"Improved pipeline:")
print(f"  Original: {len(all_chunks)} chunks (300 chars), k=4")
print(f"  Improved: {len(improved_chunks)} chunks (500 chars), k=6, stricter prompt")

In [ ]:
# Re-evaluate the improved pipeline
print("Re-evaluating improved pipeline...\n")

improved_scores = []

for i, qa in enumerate(golden_test_set, 1):
    q = qa["question"]
    gt = qa["ground_truth"]

    # Run improved pipeline
    answer = improved_chain.invoke(q)
    docs = improved_retriever.invoke(q)
    contexts = [doc.page_content for doc in docs]
    context_str = "\n".join(contexts)

    # Evaluate
    faith = evaluate_faithfulness(q, answer, context_str)
    rel = evaluate_relevancy(q, answer)
    prec = evaluate_context_precision(q, contexts)
    rec = evaluate_context_recall(gt, contexts)

    improved_scores.append({
        "faithfulness": faith,
        "relevancy": rel,
        "precision": prec,
        "recall": rec,
    })

    print(f"Q{i}: Faith={faith:.2f} Rel={rel:.2f} Prec={prec:.2f} Rec={rec:.2f}")

# Compare averages
print("\n" + "=" * 60)
print("COMPARISON: Original vs Improved")
print("=" * 60)
for metric in ["faithfulness", "relevancy", "precision", "recall"]:
    original = sum(s[metric] for s in all_scores) / len(all_scores)
    improved = sum(s[metric] for s in improved_scores) / len(improved_scores)
    delta = improved - original
    arrow = "+" if delta > 0 else ""
    print(f"  {metric:20s}  {original:.2f} -> {improved:.2f}  ({arrow}{delta:.2f})")

In [ ]:
# Before vs After comparison chart
metrics_names = ["Faithfulness", "Relevancy", "Precision", "Recall"]
metrics_keys = ["faithfulness", "relevancy", "precision", "recall"]

original_avgs = [sum(s[k] for s in all_scores) / len(all_scores) for k in metrics_keys]
improved_avgs = [sum(s[k] for s in improved_scores) / len(improved_scores) for k in metrics_keys]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, original_avgs, width, label="Original", color="#e74c3c", alpha=0.8)
bars2 = ax.bar(x + width/2, improved_avgs, width, label="Improved", color="#2ecc71", alpha=0.8)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{height:.2f}',
            ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{height:.2f}',
            ha='center', va='bottom', fontsize=10)

ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Threshold")
ax.set_ylabel("Score")
ax.set_title("RAG Pipeline: Original vs Improved")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.15)
ax.legend()
plt.tight_layout()
plt.show()

---

## YOUR TURN: Improve and Re-evaluate

Try one or more of these techniques and re-evaluate:

1. **Add HyDE** (from notebook 4.1) — generate a hypothetical answer before retrieving
2. **Add re-ranking** (from notebook 4.1) — use a cross-encoder to filter retrieved docs
3. **Change chunk size** — try 200 or 800 characters
4. **Add more test questions** — expand the golden test set with harder queries
5. **Modify the prompt** — add few-shot examples or stricter instructions

Goal: get all metrics above 0.8!

In [ ]:
# YOUR CODE HERE

# Pick one improvement technique and implement it.
# Then re-run the evaluation to see if scores improved.

# Example: Add HyDE from notebook 4.1
# hyde_prompt = ChatPromptTemplate.from_template(
#     "Write a short passage that would answer this question: {question}"
# )
# hyde_chain = hyde_prompt | llm | parser
#
# Then use the hypothetical answer for retrieval:
# hypothetical = hyde_chain.invoke({"question": question})
# docs = vectorstore.similarity_search(hypothetical, k=6)



## Key Takeaways

1. **You can't improve what you can't measure** — evaluation metrics are essential
2. **Golden test sets** require effort but are the foundation of reliable evaluation
3. **Four metrics** cover different failure modes: faithfulness, relevancy, precision, recall
4. **Diagnose before fixing** — each low metric points to a specific component
5. **Iterate** — change one thing at a time, re-evaluate, track progress

### Production Checklist
- Golden test set with 20+ questions covering all document topics
- All metrics above 0.8
- Edge cases tested (questions not in documents, ambiguous queries)
- Chunk size and overlap optimized through experimentation
- Re-ranking and/or HyDE added if retrieval metrics are low

**Congratulations!** You've completed the RAG training series. You can now build, evaluate, and improve production-quality RAG pipelines.